### Loading the Data
In this section we load the movies and ratings CSV files into pandas dataframes.

In [39]:
import pandas as pd


In [40]:
# Load the data
movies = pd.read_csv('../data/movies.csv')
ratings= pd.read_csv('../data/ratings.csv')

In [41]:
#Basic exploration
print("MOVIES")
print("-" * 20)
print(movies.shape)        # how many rows and columns
print ()
print ()
print(movies.head())       # first 5 rows
print ()
print ()
print(movies.dtypes)       # column types
print ()
print ()
print(movies.isnull().sum()) # missing values



MOVIES
--------------------
(9742, 3)


   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


movieId    int64
title        str
genres       str
dtype: object


movieId    0
title      0
genres     0
dtype: int64


In [42]:
print("Ratings")
print("-" * 20)
print(ratings.shape)
print ()
print ()
print(ratings.head())
print ()
print ()
print(ratings.dtypes)
print ()
print ()
print(ratings.isnull().sum())


Ratings
--------------------
(100836, 4)


   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931


userId         int64
movieId        int64
rating       float64
timestamp      int64
dtype: object


userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


### Creating the SQL Database
Here we create a local SQLite database and import our data into two tables: movies and ratings.

In [43]:
import sqlite3
import pandas as pd


In [44]:
# Connect to database (creates the file automatically)
# Creates a connection to a SQLite database file
conn = sqlite3.connect('../data/movies.db')
# The connection object conn is like a pipe to your database

# Load CSVs into SQL tables
movies.to_sql('movies', conn, if_exists='replace', index=False)
ratings.to_sql('ratings', conn, if_exists='replace', index=False)


100836

In [45]:
print("Database created successfully!")
print("Tables:", conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall())

conn.close()

Database created successfully!
Tables: [('movies',), ('ratings',)]


### SQL Queries & Analysis
Running the required assignment queries to explore the dataset.


In [46]:
import sqlite3
import pandas as pd
conn = sqlite3.connect('../data/movies.db')


Query 1 — Highest rated movies (min 10 ratings)

In [47]:
query_1 = """
SELECT m.title, AVG(r.rating) as avg_rating, COUNT(r.rating) as total_ratings
FROM ratings r
JOIN movies m ON r.movieId = m.movieId
GROUP BY r.movieId
HAVING total_ratings >= 10
ORDER BY avg_rating DESC
LIMIT 10
"""
highest_rated = pd.read_sql_query(query_1, conn)
print(highest_rated)

                                   title  avg_rating  total_ratings
0                  Secrets & Lies (1996)    4.590909             11
1    Guess Who's Coming to Dinner (1967)    4.545455             11
2                  Paths of Glory (1957)    4.541667             12
3       Streetcar Named Desire, A (1951)    4.475000             20
4       Celebration, The (Festen) (1998)    4.458333             12
5                             Ran (1985)    4.433333             15
6       Shawshank Redemption, The (1994)    4.429022            317
7                 His Girl Friday (1940)    4.392857             14
8  All Quiet on the Western Front (1930)    4.350000             10
9                    Hustler, The (1961)    4.333333             18


Query 2 — Most popular genres


In [48]:
query_2 = """
SELECT genres, COUNT(*) as movie_count
FROM movies
GROUP BY genres
ORDER BY movie_count DESC
LIMIT 10
"""

popular_genres = pd.read_sql_query(query_2, conn)
print(popular_genres)

                 genres  movie_count
0                 Drama         1053
1                Comedy          946
2          Comedy|Drama          435
3        Comedy|Romance          363
4         Drama|Romance          349
5           Documentary          339
6  Comedy|Drama|Romance          276
7        Drama|Thriller          168
8                Horror          167
9       Horror|Thriller          135


Query 3 — Average rating per genre


In [49]:
query_3 = """
SELECT m.genres, AVG(r.rating) as avg_rating
FROM ratings r
JOIN movies m ON r.movieId = m.movieId
GROUP BY m.genres
ORDER BY avg_rating DESC
LIMIT 10
"""
avg_per_genre = pd.read_sql_query(query_3, conn)
print(avg_per_genre)

                                 genres  avg_rating
0               Fantasy|Mystery|Western         5.0
1                  Drama|Horror|Romance         5.0
2  Drama|Fantasy|Musical|Mystery|Sci-Fi         5.0
3                 Comedy|Horror|Mystery         5.0
4  Comedy|Drama|Fantasy|Mystery|Romance         5.0
5                  Comedy|Crime|Fantasy         5.0
6             Comedy|Crime|Drama|Horror         5.0
7           Animation|Drama|Sci-Fi|IMAX         5.0
8       Animation|Drama|Fantasy|Mystery         5.0
9                 Animation|Crime|Drama         5.0


Query 4 — Top active users


In [50]:
query_4 = """
SELECT userId, COUNT(rating) as total_ratings
FROM ratings
GROUP BY userId
ORDER BY total_ratings DESC
LIMIT 10
"""
top_users = pd.read_sql_query(query_4, conn)
print(top_users)

   userId  total_ratings
0     414           2698
1     599           2478
2     474           2108
3     448           1864
4     274           1346
5     610           1302
6      68           1260
7     380           1218
8     606           1115
9     288           1055


Query 5 — Movies released by year


In [51]:
query_5 = """
SELECT 
    SUBSTR(title, LENGTH(title)-4, 4) as year,
    COUNT(*) as movie_count
FROM movies
WHERE title LIKE '%(____)'
GROUP BY year
ORDER BY year DESC
LIMIT 15
"""
movies_by_year = pd.read_sql_query(query_5, conn)
print(movies_by_year)

    year  movie_count
0   2018           41
1   2017          147
2   2016          218
3   2015          274
4   2014          277
5   2013          239
6   2012          232
7   2011          252
8   2010          247
9   2009          282
10  2008          268
11  2007          282
12  2006          295
13  2005          273
14  2004          279


In [52]:
conn.close()


## 📊 Descriptive Statistics & Data Analysis

In [53]:
# Reconnect to database and reload data:

import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/movies.db')

movies = pd.read_sql_query("SELECT * FROM movies", conn)
ratings= pd.read_sql_query("SELECT * FROM ratings", conn)

# same data as when loading the dataset

conn.close()


In [ ]:
# Basic Statistics

print("=== RATINGS STATISTICS ===")
print(ratings['rating'].describe())

print(f"\nMost common rating: {ratings['rating'].mode()[0]}") 
# [0] = takes the first one (mode can return multiple values)
print(f"Total unique users: {ratings['userId'].nunique()}")
print(f"Total unique movies rated: {ratings['movieId'].nunique()}")


=== RATINGS STATISTICS ===
count    100836.000000
mean          3.501557
std           1.042529
min           0.500000
25%           3.000000
50%           3.500000
75%           4.000000
max           5.000000
Name: rating, dtype: float64

Most common rating: 4.0
Total unique users: 610
Total unique movies rated: 9724


In [ ]:
# Movies per Genre
#But we need to split them to count indiviually
all_genres = movies['genres'].str.split('|').explode()
genre_counts = all_genres.value_counts()

# This .explode() is important — it takes Action|Comedy|Drama and turns it into 3 separate rows so you can count each genre properly.
print("=== GENRE DISTRIBUTION ===")
print(genre_counts)


=== GENRE DISTRIBUTION ===
genres
Drama                 4361
Comedy                3756
Thriller              1894
Action                1828
Romance               1596
Adventure             1263
Crime                 1199
Sci-Fi                 980
Horror                 978
Fantasy                779
Children               664
Animation              611
Mystery                573
Documentary            440
War                    382
Musical                334
Western                167
IMAX                   158
Film-Noir               87
(no genres listed)      34
Name: count, dtype: int64


In [ ]:
# Average rating per movie (merged):
avg_ratings = ratings.groupby('movieId')['rating'].agg(['mean', 'count'])

print(avg_ratings)
# now rename the columns mean and count to :
avg_ratings.columns = ['avg_rating', 'total_ratings']
avg_ratings = avg_ratings.reset_index()

# reset_index() is needed because when we use groupby(), movieId becomes the index (row labels) instead of a regular column.
# So after reset_index() movieId becomes  a normal column)
# Next we merge with movie titles
movies_with_ratings = movies.merge(avg_ratings, on='movieId')


             mean  count
movieId                 
1        3.920930    215
2        3.431818    110
3        3.259615     52
4        2.357143      7
5        3.071429     49
...           ...    ...
193581   4.000000      1
193583   3.500000      1
193585   3.500000      1
193587   3.500000      1
193609   4.000000      1

[9724 rows x 2 columns]
      movieId  avg_rating  total_ratings
0           1    3.920930            215
1           2    3.431818            110
2           3    3.259615             52
3           4    2.357143              7
4           5    3.071429             49
...       ...         ...            ...
9719   193581    4.000000              1
9720   193583    3.500000              1
9721   193585    3.500000              1
9722   193587    3.500000              1
9723   193609    4.000000              1

[9724 rows x 3 columns]


In [69]:
ratings.head()


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
